# C4: Boosting Ensemble Methods

## Introduction

**Boosting** is a family of ensemble methods that train models sequentially, where each subsequent model attempts to correct the errors made by the previous ones. Unlike Bagging (which trains models in parallel and reduces variance), Boosting focuses on reducing **Bias**.

### Bagging vs Boosting

| Feature | Bagging | Boosting |
|---|---|---|
| Training | Parallel | Sequential |
| Focus | Reduce Variance | Reduce Bias |
| Model Weights | All equal | Weighted by performance |
| Data Sampling | Bootstrap (with replacement) | Often uses full data, reweights samples |
| Example | Random Forest | AdaBoost, Gradient Boosting |

![Bagging vs Boosting](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftupleblog.github.io%2Fimages%2Fpost%2Fbaggingboosting%2Fschematic.png&f=1&nofb=1&ipt=60b017cda394ca38f520a3e48c68ad7ec1b1043bccf1a05ac02bdad0b1c1f9dc)
*(Source: ResearchGate - Schematic diagram of Bagging and Boosting methods)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, r2_score

np.random.seed(42)
print("Libraries imported.")

## 2. AdaBoost (Adaptive Boosting)

AdaBoost was one of the first boosting algorithms. It works by:

1.  **Initializing Weights**: All training samples start with equal weights.
2.  **Training a Weak Learner**: A simple model (often a Decision Stump - a tree with one split) is trained.
3.  **Calculating Error and "Amount of Say"**: The model's weighted error is calculated. A better model gets more "say" (weight) in the final prediction.
4.  **Reweighting Samples**: Misclassified samples get higher weights, forcing the next model to focus on them.
5.  **Repeat**: Steps 2-4 are repeated for `n_estimators` iterations.
6.  **Final Prediction**: A weighted vote of all weak learners.

![AdaBoost Process](https://substackcdn.com/image/fetch/$s_!_sOz!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F3ed1ed6d-2387-47f9-817e-26cfd74843ce_2667x1939.png)
*(Source: Medium - AdaBoost Process Visualization)*

In [ ]:
# Load Dataset
df_cancer = pd.read_csv('../../00-data/wisconsin_breat_cancer.csv')
if 'diagnosis' in df_cancer.columns:
    df_cancer['diagnosis'] = df_cancer['diagnosis'].map({'M': 1, 'B': 0})
    X = df_cancer.drop(['id', 'diagnosis', 'Unnamed: 32'], axis=1, errors='ignore')
    y = df_cancer['diagnosis']
else:
    X = df_cancer.iloc[:, 2:]
    y = df_cancer.iloc[:, 1].map({'M': 1, 'B': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train AdaBoost Classifier
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Decision Stump
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)
ada_clf.fit(X_train, y_train)

# Evaluate
y_pred_ada = ada_clf.predict(X_test)
print(f"AdaBoost Test Accuracy: {accuracy_score(y_test, y_pred_ada):.4f}")

## 3. Gradient Boosting

Gradient Boosting is a generalization of boosting that uses gradient descent to minimize a loss function.

1.  **Initial Prediction**: Start with a simple prediction (e.g., the mean of target values for regression).
2.  **Calculate Residuals**: Compute the difference between actual and predicted values.
3.  **Fit a Weak Learner to Residuals**: Train a decision tree to predict these residuals.
4.  **Update Predictions**: Add the new tree's predictions (scaled by a learning rate) to the current model.
5.  **Repeat**: Steps 2-4 are repeated for `n_estimators` iterations.

The "gradient" refers to the use of gradients of the loss function to guide the learning process.

![Gradient Boosting](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fpythongeeks.org%2Fwp-content%2Fuploads%2F2022%2F03%2Fworking-of-gradient-boosting-algorithm.webp&f=1&nofb=1&ipt=43ca2d33c5ebd1d0923bc1af38861ffd6f7c31ccdcabf9c98f8776cea88a74a4)
*(Source: Medium - Gradient Boosting Residual Fitting)*

In [ ]:
# Train Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

# Evaluate
y_pred_gb = gb_clf.predict(X_test)
print(f"Gradient Boosting Test Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")

### Modern Gradient Boosting Libraries

Several highly optimized libraries have been built on top of the Gradient Boosting concept:

-   **XGBoost (Extreme Gradient Boosting)**: Very popular for Kaggle competitions.
-   **LightGBM (Light Gradient Boosting Machine)**: Developed by Microsoft, known for speed.
-   **CatBoost**: Developed by Yandex, handles categorical features natively.

These libraries offer significant speed and performance improvements over `sklearn`'s `GradientBoostingClassifier`.

## 4. Comparing AdaBoost and Gradient Boosting

In [ ]:
# Cross-Validation Comparison
ada_scores = cross_val_score(ada_clf, X, y, cv=5, scoring='accuracy')
gb_scores = cross_val_score(gb_clf, X, y, cv=5, scoring='accuracy')

print(f"AdaBoost CV Accuracy: {ada_scores.mean():.4f} (+/- {ada_scores.std():.4f})")
print(f"Gradient Boosting CV Accuracy: {gb_scores.mean():.4f} (+/- {gb_scores.std():.4f})")

## 5. Exercise: Gradient Boosting Regressor

Apply Gradient Boosting to a regression problem: predicting `mpg` from the Auto MPG dataset.

In [ ]:
# Load Data
df_mpg = pd.read_csv('../../00-data/auto_mpg.csv')
df_mpg = df_mpg.replace('?', np.nan).dropna()
X_reg = pd.get_dummies(df_mpg.drop(columns=['mpg', 'car name'], errors='ignore'), drop_first=True, dtype=float)   # this copy's origin is text
y_reg = df_mpg['mpg']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# TODO: Train a GradientBoostingRegressor with 100 estimators.

# TODO: Evaluate R2 score on test data.

# TODO: (Optional) Plot feature importances using gb_reg.feature_importances_